In [0]:
#Broadcast variable = read‑only shared variable sent to all executors only once.
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("BroadcastExample").getOrCreate()

# Small lookup table
lookup = {
    "A": "Apple",
    "B": "Banana",
    "C": "Cherry"
}

broadcast_lookup = spark.sparkContext.broadcast(lookup)

# Big dataset
data = ["A", "B", "C", "A", "B", "C"]
rdd = spark.sparkContext.parallelize(data)

def map_values(x):
    fruit = broadcast_lookup.value.get(x)
    return (x, fruit)

result = rdd.map(map_values).collect()
print(result)


#Expected Output Need to map the value for ["A", "B", "C", "A", "B", "C"] 
#(["A", "Apple"], ["B", "Banana"], ["C", "Cherry"], ["A", "Apple"], ["B", "Banana"], ["C", "Cherry"])

In [0]:
# Accumulator = a write‑only shared variable used for counters or sums across executors.  
# Workers can add to it, but cannot read it.
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("AccumulatorExample").getOrCreate()
acc = spark.sparkContext.accumulator(0)
data = [1, 2, 3, 4, 5]
rdd = spark.sparkContext.parallelize(data)

def add_if_even(x):
    if x % 2 == 0:
        acc.add(x)
    return x

rdd.map(add_if_even).collect()
print("Total even numbers:", acc.value)

In [0]:
# UDF = Your own Python function used inside Spark SQL/DataFrame operations.  
# Used when Spark built‑in functions are not enough.

# ⚠️ UDFs are slower because they break Spark’s optimization.



from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, lit,current_date
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("UDFExample").getOrCreate()
df = spark.createDataFrame([(1,), (2,), (3,)], ["num"])

def square(x):
    return x * 2

square_udf = udf(square)
df2 = df.withColumn("double_num", square_udf(df.num))
df2 = df2.withColumn("Dept", lit("Sales"))
df2=df2.withColumn("Currentdate",current_date())
display(df2)

num,double_num,Dept,Currentdate
1,2,Sales,2026-05-11
2,4,Sales,2026-05-11
3,6,Sales,2026-05-11


In [0]:
from pyspark.sql import Row

# Sample DataFrames with different columns
df1 = spark.createDataFrame([Row(id=1, name="Alice"), Row(id=2, name="Bob")])
df2 = spark.createDataFrame([Row(id=3, age=30), Row(id=4, age=25)])


display(df1)
display(df2)
# Union by name, allowing missing columns
df_union = df1.unionByName(df2, allowMissingColumns=True)
display(df_union)

id,name
1,Alice
2,Bob


id,age
3,30
4,25


id,name,age
1,Alice,null
2,Bob,null
3,null,30
4,null,25


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col

# Create the DataFrame
data = [
    Row(Team1="India", Team2="Sri Lanka", Winner="India"),
    Row(Team1="Australia", Team2="India", Winner="Australia"),
    Row(Team1="Australia", Team2="Sri Lanka", Winner="Australia"),
    Row(Team1="New Zealand", Team2="Sri Lanka", Winner="Sri Lanka"),
    Row(Team1="New Zealand", Team2="Australia", Winner="New Zealand"),
    Row(Team1="New Zealand", Team2="India", Winner="India")
]
df_match = spark.createDataFrame(data)

# Count matches played by each team
df_tot1 = df_match.select(col("Team1").alias("Country"))
df_tot2 = df_match.select(col("Team2").alias("Country"))
df_country = df_tot1.union(df_tot2)
df_final = df_country.groupBy("Country").count().withColumnRenamed("count", "Total")

# Count matches won by each team
df_winning = df_match.groupBy("Winner").count().withColumnRenamed("Winner", "Country").withColumnRenamed("count", "Winning_cnt")

# Join and display
df_final_output = df_final.join(df_winning, on="Country", how="left").select("Country", "Total", "Winning_cnt")
display(df_final_output)

Country,Total,Winning_cnt
India,3,2
Australia,3,2
New Zealand,3,1
Sri Lanka,3,1


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import greatest, max

# Create the DataFrame
data = [
    Row(Col1=3, Col2=4, Col3=5),
    Row(Col1=8, Col2=7, Col3=6),
    Row(Col1=1, Col2=6, Col3=3)
]
df1 = spark.createDataFrame(data)
col=df1.columns
# Add a column with the max value across Col1, Col2, Col3 for each row
df1 = df1.withColumn("MaxValue", greatest(*col))
display(df1)
df1 = df1.select(max("MaxValue"))
display(df1)

Col1,Col2,Col3,MaxValue
3,4,5,5
8,7,6,8
1,6,3,6


max(MaxValue)
8


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import when, col, max

# Create df1
data = [
    Row(Empid=101, name="David", year=2019, salary=20000),
    Row(Empid=101, name="David", year=2020, salary=30000),
    Row(Empid=102, name="guru", year=2019, salary=50000),
    Row(Empid=102, name="guru", year=2020, salary=65000)
]
df1 = spark.createDataFrame(data)
display(df1)

df4 = df1.groupBy("Empid", "name").agg(max("year").alias("Recent_Year"))
display(df4)

# Join with aliases and use qualified column references
df1_alias = df1.alias("a")
df4_alias = df4.alias("b")

df_joined = df1_alias.join(df4_alias, (col("a.Empid") == col("b.Empid")) & (col("a.name") == col("b.name")), "inner") \
           .select(col("a.Empid"), col("a.name"), col("a.year"), col("a.salary"), col("b.Recent_Year"))

df_result = df_joined.withColumn(
    "diff_sal",
    when(col("year") != col("Recent_Year"), -col("salary")).otherwise(col("salary"))
).select(col("Empid"), col("name"), col("year"), col("salary"), col("Recent_Year"), col("diff_sal"))

display(df_result)

Empid,name,year,salary
101,David,2019,20000
101,David,2020,30000
102,guru,2019,50000
102,guru,2020,65000


Empid,name,Recent_Year
101,David,2020
102,guru,2020


Empid,name,year,salary,Recent_Year,diff_sal
101,David,2019,20000,2020,-20000
101,David,2020,30000,2020,30000
102,guru,2019,50000,2020,-50000
102,guru,2020,65000,2020,65000


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import explode, col

data = [
    Row(Name='Abhishek', Age='20', Skills=['Big Data', 'Spark']),
    Row(Name='Vivek', Age='21', Skills=['ML', 'AI']),
    Row(Name='Rohit', Age='19', Skills=None)
]

df1 = spark.createDataFrame(data)
df2 = df1.select("Name", "Skills").withColumn("split_Skills", explode(col("Skills")))
display(df2)

Name,Skills,split_Skills
Abhishek,"List(Big Data, Spark)",Big Data
Abhishek,"List(Big Data, Spark)",Spark
Vivek,"List(ML, AI)",ML
Vivek,"List(ML, AI)",AI


In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import lag, col
from pyspark.sql.window import Window

# Create Weather DataFrame
data = [
    Row(id=1, recordDate="2015-01-01", temperature=10),
    Row(id=2, recordDate="2015-01-02", temperature=25),
    Row(id=3, recordDate="2015-01-03", temperature=20),
    Row(id=4, recordDate="2015-01-04", temperature=30)
]
df_weather = spark.createDataFrame(data)

# Define window specification
windowSpec = Window.orderBy("recordDate")

# Add previous day's temperature using lag
df_weather_lag = df_weather.withColumn(
    "prev_temp", lag("temperature").over(windowSpec)
)

display(df_weather_lag)